### Setup

In [ ]:
from pathlib import Path
import mgnify_methods.paper_modules as pm
from mgnify_methods.taxonomy import (
    pivot_taxonomic_data,
    aggregate_by_taxonomic_level,
)

from mgnify_methods.taxonomy import (
    remove_singletons_per_sample,
    prevalence_cutoff_abund,
)

from mgnify_methods.metacomp.transforms import (
    apply_transform_method,
)

In [ ]:
root_dir = Path().resolve().parent
contig_path = root_dir / "configs" / "model_test.json"

CONFIG = pm.config_setup(root_dir, contig_path)

### Preprocess EMO-BON data

In [ ]:
# loads both ssu and emobon metadata, 
abundance_emobon, emobon_meta = pm.load_emobon(root_dir, ret='ssu')
abundance_emobon.shape, emobon_meta.shape

In [ ]:
abundance_emobon = pm.process_emobon_data(abundance_emobon, CONFIG)
abundance_emobon.shape

In [ ]:
# turn abundance into taxonomy for cleaning lineages
taxonomy_table = pm.clean_abundance_table(abundance_emobon, CONFIG)

tax_level = CONFIG['taxonomy']['analysis_level']
abundance_table = aggregate_by_taxonomic_level(
    taxonomy_table.df,
    level=tax_level,
    dropna=CONFIG['preprocessing']['dropna']
)
del taxonomy_table

In [ ]:
abundance_table = pivot_taxonomic_data(abundance_table)
type(abundance_table)

## Processing of the raw abundance table
- remove singletons
- prevalence cutoff
- CLR

In [ ]:
abundance_table_beta = remove_singletons_per_sample(abundance_table, skip_columns=0)

abundance_table_beta = prevalence_cutoff_abund(
        abundance_table, 
        percent=CONFIG['preprocessing']['prevalence_cutoff'],
        skip_columns=0
    )

preprocess_tables = {'emobon': abundance_table}

## Data transformation for beta diversity
if CONFIG['transform']['enabled']:
    transformed_tables = {}
    for sample_type, table_or_dict in preprocess_tables.items():
        if isinstance(table_or_dict, dict):
            transformed_tables[sample_type] = {
                tax_level: apply_transform_method(df, CONFIG)
                for tax_level, df in table_or_dict.items()
            }
        else:
            transformed_tables[sample_type] = apply_transform_method(table_or_dict, CONFIG)

    preprocess_tables = transformed_tables

## Modeling from scratch
- unrealistically simple example

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

In [ ]:
# filter metadata columns specified in config
emobon_meta_in = emobon_meta[CONFIG['modeling']['metadata_cols']].copy()

# abundance table is taxa x samples in this workflow; transpose to samples x taxa
abundance_for_model = preprocess_tables["emobon"].transpose(copy=False)

# align by shared sample IDs
shared_ids = emobon_meta_in.index.intersection(abundance_for_model.index)
X_raw = emobon_meta_in.loc[shared_ids].copy()
y_raw = abundance_for_model.loc[shared_ids].copy()

In [ ]:
# group-aware split + simple preprocessing
groups = X_raw['observatory ID'].astype('string')

# keep 'observatory ID' only for split grouping, not as predictive feature
X_features = X_raw.drop(columns=['observatory ID'])
X_features = pd.get_dummies(X_features, dummy_na=False)

# drop rows with any missing values for simplicity
valid_mask = (
    X_features.notna().all(axis=1)
    & y_raw.notna().all(axis=1)
    & groups.notna()
    & groups.astype(str).ne('<NA>')
)

X = X_features.loc[valid_mask]
y = y_raw.loc[valid_mask]
groups = groups.loc[valid_mask]

logo = LeaveOneGroupOut()
len(X), len(y), groups.nunique()

In [ ]:
reg = RandomForestRegressor(
    n_estimators=1000,
    max_depth=None,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
)

In [ ]:
# fit and evaluate per held-out group
fold_rmse = []

for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    reg.fit(X_train, y_train)
    y_pred = reg.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test.to_numpy(), y_pred))
    fold_rmse.append(rmse)

float(np.mean(fold_rmse)), float(np.std(fold_rmse))

## Modeling using this package

In [ ]:
from emobon_models.modeling_config import modeling_config_from_analysis
from emobon_models.modeling_runner import run_group_loocv_with_mlflow

In [ ]:

modeling_config = modeling_config_from_analysis(CONFIG)

modeling_results = run_group_loocv_with_mlflow(
    metadata_df=emobon_meta_in,
    abundance_df=abundance_for_model,
    config=modeling_config,
)

modeling_results["summary_metrics"]
